# Esta parte do código se refere à pipeline da camada SILVER em BATCH para testes antes de subir ao AWS

In [103]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
# Instalando as dependências
# ~~~~~~~~~~~~~~~~~~~~~~~~~~

# pyarrow para salvar em PARQUET

!pip install pyarrow colorama tabulate --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\carol\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [104]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
# Importações
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
import logging
from pathlib import Path
from datetime import datetime

import pandas as pd

In [105]:
# ~~~~~~~~~~~~~~~
# CONFIGURAÇÕES
# ~~~~~~~~~~~~~~~
DATA_BRONZE = Path("bronze")
DATA_SILVER = Path("silver")

DATA_SILVER.mkdir(parents=True, exist_ok=True)

INGESTION_DATE = datetime.now().strftime("%Y-%m-%d")

TABELAS = [
    "uf",
    "meta_alfabetizacao_brasil",
    "meta_alfabetizacao_uf",
    "meta_alfabetizacao_municipio",
    "municipio",
    "alunos"
]

In [106]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
# CONFIGURAÇÃO DOS LOGS
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s"
)

log = logging.getLogger(__name__)

In [107]:
# ~~~~~~~~~~~~~~~
# LOG INICIAL
# ~~~~~~~~~~~~~~~

log.info("~" * 35)
log.info("INICIANDO ETL DA CAMADA SILVER")
log.info("~" * 35)

2026-07-12 20:51:14,050 | INFO     | ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
2026-07-12 20:51:14,051 | INFO     | INICIANDO ETL DA CAMADA SILVER
2026-07-12 20:51:14,051 | INFO     | ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# LENDO ARQUIVOS DA CAMADA BRONZE
"""
    Lê o arquivo Parquet mais recente da camada Bronze para uma tabela.

    A Bronze agora particiona por ingestion_date (bronze/{tabela}/
    ingestion_date={data}/{tabela}.parquet), preservando o histórico de
    todas as cargas. A Silver processa, por padrão, a partição mais
    recente (a última carga executada).

    Args:
        tabela (str): Nome da tabela.
"""
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
def ler_bronze(tabela):

    pasta_tabela = DATA_BRONZE / tabela

    particoes = sorted(pasta_tabela.glob("ingestion_date=*"))

    if not particoes:
        raise FileNotFoundError(
            f"Nenhuma partição de ingestion_date encontrada em {pasta_tabela}. "
            f"Rode a camada Bronze antes da Silver."
        )

    particao_mais_recente = particoes[-1]
    caminho = particao_mais_recente / f"{tabela}.parquet"

    log.info(f"Lendo a camada Bronze (partição mais recente): {caminho}")

    return pd.read_parquet(caminho)

In [109]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# REGRAS DE CHECKS 
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

CHECKS = {

    "uf": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "ano", "critico": True},
        {"tipo": "not_null", "coluna": "sigla_uf", "critico": True},
        {"tipo": "not_null", "coluna": "sigla_uf_nome", "critico": True},
        {"tipo": "unique", "coluna": ["ano","sigla_uf","serie","rede"], "critico": False},
    ],

    "municipio": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "ano", "critico": True},
        {"tipo": "not_null", "coluna": "id_municipio", "critico": True},
        {"tipo": "not_null", "coluna": "id_municipio_nome", "critico": True},
        {"tipo": "unique", "coluna": ["ano","id_municipio","serie","rede"], "critico": False},
    ],

    "meta_alfabetizacao_brasil": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "ano", "critico": True},
        {"tipo": "not_null", "coluna": "rede", "critico": True},
    ],

    "meta_alfabetizacao_uf": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "ano", "critico": True},
        {"tipo": "not_null", "coluna": "sigla_uf", "critico": True},
        {"tipo": "not_null", "coluna": "sigla_uf_nome", "critico": True},
        {"tipo": "unique", "coluna": ["ano","sigla_uf","rede"], "critico": False},
    ],

    "meta_alfabetizacao_municipio": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "ano", "critico": True},
        {"tipo": "not_null", "coluna": "id_municipio", "critico": True},
        {"tipo": "not_null", "coluna": "id_municipio_nome", "critico": True},
        {"tipo": "unique", "coluna": ["ano","id_municipio","rede"], "critico": False}
    ],

    "alunos": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "id_aluno", "critico": True},
        {"tipo": "not_null", "coluna": "id_escola", "critico": True},
        {"tipo": "not_null", "coluna": "id_municipio", "critico": True},
    ]
}

In [110]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# CHECANDO A QUALIDADE DOS DADOS DA SILVER
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def checar_qualidade(df, checks):
    """
    Executa as validações de qualidade da camada Silver.

    Args:
        df (pandas.DataFrame): DataFrame da Silver.
        checks (list): Lista de regras de validação.

    Raises:
        Exception: Caso alguma validação crítica falhe.
    """

    log.info(f"[DQ:SILVER] Iniciando verificações ({len(checks)} regra(s))")

    passou = 0
    falhou = 0
    criticos = 0

    for check in checks:

        tipo = check["tipo"]
        coluna = check.get("coluna")
        valor = check.get("valor")
        critico = check.get("critico", True)

        ok = False
        detalhe = ""

        try:

            if tipo == "not_null":

                nulos = df[coluna].isnull().sum()

                ok = nulos == 0
                detalhe = f"{nulos} nulos encontrados"

            elif tipo == "min_count":

                contagem = len(df)

                ok = contagem >= valor
                detalhe = f"contagem={contagem} | mínimo={valor}"

            elif tipo == "unique":

                duplicados = df.duplicated(subset=coluna).sum()

                ok = duplicados == 0
                detalhe = f"{duplicados} duplicados encontrados"

        except Exception as e:

            ok = False
            detalhe = str(e)

        status = "PASS" if ok else ("FAIL" if critico else "WARN")

        if ok:

            passou += 1
            log.info(f"[DQ:SILVER] {status} | {tipo} | {coluna} | {detalhe}")

        else:

            falhou += 1

            if critico:

                criticos += 1
                log.error(f"[DQ:SILVER] {status} | {tipo} | {coluna} | {detalhe}")

            else:

                log.warning(f"[DQ:SILVER] {status} | {tipo} | {coluna} | {detalhe}")

    score = round((passou / len(checks)) * 100, 1)

    log.info(f"[DQ:SILVER] Score={score}% | PASS={passou} | FAIL={falhou}")

    if criticos > 0:

        raise Exception(
            f"[DQ:SILVER] {criticos} validação(ões) crítica(s) falharam."
        )

In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# SCHEMA DE TIPOS POR TABELA (conversão de tipo da Silver)
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

_COLUNAS_META = {f"meta_alfabetizacao_20{ano}": "float64" for ano in range(24, 31)}
_COLUNAS_PROPORCAO = {f"proporcao_aluno_nivel_{n}": "float64" for n in range(0, 9)}

TIPOS_COLUNAS = {
    "uf": {
        "ano": "Int64", "sigla_uf": "string", "sigla_uf_nome": "string",
        "serie": "string", "rede": "string",
        "taxa_alfabetizacao": "float64", "media_portugues": "float64",
        **_COLUNAS_PROPORCAO,
    },
    "municipio": {
        "ano": "Int64", "id_municipio": "Int64", "id_municipio_nome": "string",
        "serie": "string", "rede": "string",
        "taxa_alfabetizacao": "float64", "media_portugues": "float64",
        **_COLUNAS_PROPORCAO,
    },
    "meta_alfabetizacao_brasil": {
        "ano": "Int64", "rede": "string", "taxa_alfabetizacao": "float64",
        "percentual_participacao": "float64",
        **_COLUNAS_META,
    },
    "meta_alfabetizacao_uf": {
        "ano": "Int64", "sigla_uf": "string", "sigla_uf_nome": "string",
        "rede": "string", "taxa_alfabetizacao": "float64",
        "percentual_participacao": "float64",
        **_COLUNAS_META,
    },
    "meta_alfabetizacao_municipio": {
        "ano": "Int64", "id_municipio": "Int64", "id_municipio_nome": "string",
        "rede": "string", "taxa_alfabetizacao": "float64",
        "nivel_alfabetizacao": "string", "percentual_participacao": "float64",
        **_COLUNAS_META,
    },
    "alunos": {
        "ano": "Int64", "id_municipio": "Int64", "id_municipio_nome": "string",
        "id_escola": "Int64", "id_aluno": "Int64",
        "caderno": "string", "serie": "string", "rede": "string",
        "presenca": "string", "preenchimento_caderno": "string",
        "alfabetizado": "string", "proficiencia": "float64", "peso_aluno": "float64",
    },
}

# Colunas que funcionam como chave de junção entre tabelas e cujo *formato*
# (não o conteúdo/categoria) precisa ser padronizado para o merge funcionar
# de forma confiável. `rede`/`serie` NÃO entram aqui: o notebook da Gold
# faz comparação de string literal (ex.: `df["rede"] != "Total (...)"`),
# então alterar caixa/formatação delas quebraria esse filtro silenciosamente.
COLUNAS_CHAVE_NORMALIZAR = ["sigla_uf"]


def aplicar_conversao_tipo(df, tabela):
    """
    Converte cada coluna para o tipo declarado em TIPOS_COLUNAS, evitando que
    chaves de junção (ano, id_municipio) fiquem com tipos diferentes entre
    tabelas (ex.: int64 numa tabela e float64/string noutra), o que faz o
    merge falhar silenciosamente (não dá erro, só não casa nenhuma linha).
    """
    tipos = TIPOS_COLUNAS.get(tabela, {})

    for coluna, tipo in tipos.items():

        if coluna not in df.columns:
            continue

        try:
            if tipo in ("Int64", "float64"):
                df[coluna] = pd.to_numeric(df[coluna], errors="coerce").astype(tipo)
            else:
                df[coluna] = df[coluna].astype("string")
        except Exception as e:
            log.warning(f"[SILVER] {tabela}: falha ao converter '{coluna}' para {tipo}: {e}")

    return df


def normalizar_chaves_join(df, tabela):
    """
    Padroniza o formato (maiúsculas/espaços) das colunas-chave de junção
    territoriais. Mantém o conteúdo de colunas categóricas de negócio
    (rede, serie) intacto, pois a Gold depende do texto exato delas.
    """
    for coluna in COLUNAS_CHAVE_NORMALIZAR:
        if coluna in df.columns:
            df[coluna] = df[coluna].str.strip().str.upper()

    return df

In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# COLUNAS ESSENCIAIS POR TABELA (usadas no tratamento de nulos e como chave de dedup)
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
COLUNAS_ESSENCIAIS = {
    "uf": ["ano", "sigla_uf"],
    "municipio": ["ano", "id_municipio"],
    "meta_alfabetizacao_brasil": ["ano", "rede"],
    "meta_alfabetizacao_uf": ["ano", "sigla_uf"],
    "meta_alfabetizacao_municipio": ["ano", "id_municipio"],
    "alunos": ["id_aluno", "id_escola", "id_municipio"],
}

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# APLICA AS TRANSFORMAÇÕES COMUNS DA CAMADA SILVER
"""
    Args:
        df (pandas.DataFrame): DataFrame da Bronze.
        tabela (str): Nome da tabela.

    Nota sobre rastreabilidade:
        Os metadados técnicos da Bronze (_record_hash, _source_dataset,
        _source_table, _ingestion_timestamp, _ingestion_date) NÃO são
        removidos aqui. Eles atravessam a Silver propositalmente, para
        permitir auditoria de ponta a ponta (padrão Medalhão). A limpeza
        do schema analítico acontece na Gold, que já seleciona
        explicitamente suas colunas finais.
"""
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
def construir_silver(df, tabela):

    log.info(f"Transformando tabela: {tabela}")

    df = df.copy()

    linhas_antes = len(df)

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Remove espaços em branco
    # (metadados técnicos da Bronze são preservados - ver nota acima)
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    colunas_texto = df.select_dtypes(include="object").columns

    for coluna in colunas_texto:
        df[coluna] = df[coluna].str.strip()

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Conversão de tipo
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    df = aplicar_conversao_tipo(df, tabela)

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Normalização de chave (formato, não conteúdo de negócio)
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    df = normalizar_chaves_join(df, tabela)

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Remove duplicidade por CHAVE DE NEGÓCIO (não pela linha inteira).
    # Como os metadados de ingestão agora são preservados, duas cargas do
    # mesmo registro de negócio em datas diferentes teriam
    # _ingestion_timestamp diferente e não seriam pegas por um dedup de
    # linha inteira. Por isso ordenamos por _ingestion_timestamp e
    # mantemos a versão mais recente por chave de negócio.
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    chave_negocio = COLUNAS_ESSENCIAIS.get(tabela, [])
    chave_negocio = [c for c in chave_negocio if c in df.columns]

    if chave_negocio:

        if "_ingestion_timestamp" in df.columns:
            df = df.sort_values("_ingestion_timestamp")

        antes = len(df)
        df = df.drop_duplicates(subset=chave_negocio, keep="last")
        duplicados = antes - len(df)

        if duplicados > 0:
            log.info(
                f"[SILVER] {tabela}: {duplicados} registro(s) duplicado(s) "
                f"por chave de negócio {chave_negocio} removido(s) "
                f"(mantida a versão mais recente por _ingestion_timestamp)"
            )
    else:
        antes = len(df)
        df = df.drop_duplicates()
        duplicados = antes - len(df)
        if duplicados > 0:
            log.info(f"[SILVER] {tabela}: {duplicados} linha(s) duplicada(s) (linha inteira) removida(s)")

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Trata nulos em colunas essenciais (chaves de negócio)
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    colunas_essenciais = COLUNAS_ESSENCIAIS.get(tabela, [])
    colunas_essenciais = [c for c in colunas_essenciais if c in df.columns]

    if colunas_essenciais:

        nulos_antes = len(df)
        df = df.dropna(subset=colunas_essenciais)
        removidas = nulos_antes - len(df)

        if removidas > 0:
            log.warning(
                f"[SILVER] {tabela}: {removidas} linha(s) removida(s) por nulo "
                f"em coluna essencial {colunas_essenciais}"
            )

    log.info(
        f"[SILVER] {tabela}: {linhas_antes} linha(s) na entrada -> "
        f"{len(df)} linha(s) na saída"
    )

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Adiciona metadado da Silver
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    df["_silver_processed_at"] = datetime.now()

    log.info(f"Tabela {tabela} transformada")

    return df

In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# INTEGRAÇÃO ENTRE AS BASES: ALUNOS + MUNICÍPIO + UF
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

# Mapeamento oficial do IBGE: os 2 primeiros dígitos do código de 7 dígitos
# do município identificam a UF. Usado para derivar `sigla_uf` em `alunos`,
# já que a extração da Base dos Dados não trouxe essa coluna diretamente.
CODIGO_UF_PARA_SIGLA = {
    11: "RO", 12: "AC", 13: "AM", 14: "RR", 15: "PA", 16: "AP", 17: "TO",
    21: "MA", 22: "PI", 23: "CE", 24: "RN", 25: "PB", 26: "PE", 27: "AL",
    28: "SE", 29: "BA",
    31: "MG", 32: "ES", 33: "RJ", 35: "SP",
    41: "PR", 42: "SC", 43: "RS",
    50: "MS", 51: "MT", 52: "GO", 53: "DF",
}


def derivar_sigla_uf(id_municipio):
    """
    Deriva a sigla da UF a partir do código IBGE do município.

    Args:
        id_municipio: Código IBGE do município (string ou int, 7 dígitos).

    Returns:
        str | None: Sigla da UF, ou None se o código for inválido/ausente.
    """
    try:
        codigo_uf = int(str(int(id_municipio))[:2])
        return CODIGO_UF_PARA_SIGLA.get(codigo_uf)
    except (ValueError, TypeError):
        return None


def construir_silver_alunos_integrado(df_alunos, df_municipio, df_uf):
    """
    Integra a tabela de alunos (granularidade individual) com indicadores
    agregados de município e de UF, adicionando contexto territorial a
    cada registro de aluno.

    Args:
        df_alunos (pandas.DataFrame): Tabela `alunos` já tratada na Silver.
        df_municipio (pandas.DataFrame): Tabela `municipio` já tratada na Silver.
        df_uf (pandas.DataFrame): Tabela `uf` já tratada na Silver.

    Returns:
        pandas.DataFrame: `alunos` enriquecido com contexto municipal e estadual.
    """

    log.info("Integrando alunos + município + UF")

    df = df_alunos.copy()

    # Deriva a UF do aluno a partir do código do município
    df["sigla_uf"] = df["id_municipio"].apply(derivar_sigla_uf)

    sem_uf = df["sigla_uf"].isnull().sum()
    if sem_uf > 0:
        log.warning(f"[SILVER] {sem_uf} aluno(s) sem sigla_uf derivada")

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Contexto municipal (mesmo ano + município + rede)
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    contexto_municipio = (
        df_municipio[["ano", "id_municipio", "rede", "taxa_alfabetizacao", "media_portugues"]]
        .rename(columns={
            "taxa_alfabetizacao": "taxa_alfabetizacao_municipio",
            "media_portugues": "media_portugues_municipio",
        })
    )

    df = df.merge(
        contexto_municipio,
        on=["ano", "id_municipio", "rede"],
        how="left",
    )

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Contexto estadual (mesmo ano + UF + rede)
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    contexto_uf = (
        df_uf[["ano", "sigla_uf", "sigla_uf_nome", "rede", "taxa_alfabetizacao", "media_portugues"]]
        .rename(columns={
            "taxa_alfabetizacao": "taxa_alfabetizacao_uf",
            "media_portugues": "media_portugues_uf",
        })
    )

    df = df.merge(
        contexto_uf,
        on=["ano", "sigla_uf", "rede"],
        how="left",
    )

    df["_silver_integrado_processed_at"] = datetime.now()

    log.info(
        f"Integração concluída: {len(df)} registro(s) de aluno, "
        f"{df['taxa_alfabetizacao_municipio'].isnull().sum()} sem contexto municipal, "
        f"{df['taxa_alfabetizacao_uf'].isnull().sum()} sem contexto estadual"
    )

    return df

In [112]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# SALVA UM DATAFRAME NA CAMADA SILVER EM FORMATO PARQUET
"""
    Args:
        df (pandas.DataFrame): DataFrame tratado.
        tabela (str): Nome da tabela.
"""
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
def salvar_silver(df, tabela):

    caminho = DATA_SILVER / f"{tabela}.parquet"

    df.to_parquet(
        caminho,
        index=False
    )

    log.info(f"Tabela {tabela} da camada SILVER salva em {caminho}")

    return caminho

In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# EXECUTA TODA A CAMADA SILVER PARA TODAS AS TABELAS DA BRONZE
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
def executar_silver():

    tabelas_silver = {}

    for tabela in TABELAS:

        log.info("~" * 60)
        log.info(f"Iniciando camada SILVER: {tabela}")
        log.info("~" * 60)

        df = ler_bronze(tabela)

        df_silver = construir_silver(
            df,
            tabela
        )

        checks = CHECKS.get(tabela, [])

        if checks:
            checar_qualidade(df_silver, checks)

        salvar_silver(
            df_silver,
            tabela
        )

        tabelas_silver[tabela] = df_silver

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Integração entre as bases (alunos + município + UF)
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    log.info("~" * 60)
    log.info("Iniciando integração: alunos + município + UF")
    log.info("~" * 60)

    df_alunos_integrado = construir_silver_alunos_integrado(
        tabelas_silver["alunos"],
        tabelas_silver["municipio"],
        tabelas_silver["uf"],
    )

    salvar_silver(df_alunos_integrado, "alunos_integrado")

    log.info("Camada SILVER concluída!")

In [114]:
executar_silver()

2026-07-12 20:51:14,127 | INFO     | ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
2026-07-12 20:51:14,128 | INFO     | Iniciando camada SILVER: uf
2026-07-12 20:51:14,129 | INFO     | ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
2026-07-12 20:51:14,130 | INFO     | Lendo a camada Bronze: bronze\uf.parquet
2026-07-12 20:51:14,213 | INFO     | Transformando tabela: uf
2026-07-12 20:51:14,218 | INFO     | Tabela uf transformada
2026-07-12 20:51:14,219 | INFO     | [DQ:SILVER] Iniciando verificações (5 regra(s))
2026-07-12 20:51:14,219 | INFO     | [DQ:SILVER] PASS | min_count | None | contagem=145 | mínimo=1
2026-07-12 20:51:14,220 | INFO     | [DQ:SILVER] PASS | not_null | ano | 0 nulos encontrados
2026-07-12 20:51:14,221 | INFO     | [DQ:SILVER] PASS | not_null | sigla_uf | 0 nulos encontrados
2026-07-12 20:51:14,222 | INFO     | [DQ:SILVER] PASS | not_null | sigla_uf_nome | 0 nulos encontrados
2026-07-12 20:51:14,224 | INFO     | [DQ:SILVER] PASS | uniqu